ASSIGNMENT NLP – 5 (Token Classification: POS Tagging & Chunking)


In [ ]:

# STEP 1: INSTALL LIBRARIES

!pip install transformers datasets seqeval evaluate -q



# STEP 2: IMPORT LIBRARIES

import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
import evaluate



# STEP 3: LOAD DATASET (POS TAGGING)

dataset = load_dataset("universal_dependencies", "en_ewt")

print("Sample Data:")
print(dataset["train"][0])


# STEP 4: LABELS

label_list = dataset["train"].features["upos"].feature.names
print("\nLabels:")
print(label_list)


# STEP 5: TOKENIZER

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")



# STEP 6: TOKENIZATION + LABEL ALIGNMENT

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["upos"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)



# STEP 7: MODEL SETUP

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(label_list),
    id2label={i: label for i, label in enumerate(label_list)},
    label2id={label: i for i, label in enumerate(label_list)}
)



# STEP 8: TRAINING ARGUMENTS

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
)



# STEP 9: EVALUATION METRIC
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }



# STEP 10: TRAINER

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)



# STEP 11: TRAIN MODEL

trainer.train()



# STEP 12: EVALUATE MODEL

results = trainer.evaluate()
print("\nEvaluation Results:")
print(results)



# STEP 13: INFERENCE FUNCTION

def predict(sentence):
    tokens = sentence.split()
    inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

    outputs = model(**inputs)
    predictions = np.argmax(outputs.logits.detach().numpy(), axis=2)

    word_ids = inputs.word_ids()
    result = []

    for i, word_idx in enumerate(word_ids):
        if word_idx is not None:
            label = label_list[predictions[0][i]]
            result.append((tokens[word_idx], label))

    return result



# STEP 14: TEST INFERENCE

sentence = "John works at Google in California"
print("\nPrediction:")
print(predict(sentence))



# STEP 15: THEORY (WRITE IN MARKDOWN CELL)

"""
COMPARISON:

POS Tagging:
- Labels each word (NOUN, VERB, etc.)
- Easier task

Chunking:
- Groups words into phrases (NP, VP)
- More complex

CHALLENGES:
- Subword tokenization
- Label alignment (-100)
- Training time

OBSERVATIONS:
- BERT gives high accuracy
- Context improves performance
"""